In [ ]:
pip install faiss-cpu sentence-transformers pandas
# Install required libraries for vector search and embeddings

In [ ]:
import pandas as pd

# Load the dataset from Google Drive
# The dataset contains two main columns: 'question' and 'answer'
# This is the normalized dataset prepared during data cleaning phase

df = pd.read_csv("/content/drive/MyDrive/llama model/normalized_ds_questions.csv")


# Convert DataFrame columns into Python lists
# These lists will be used for model training or processing
questions = df["question"].tolist()
answers = df["answer"].tolist()

In [ ]:
# This step loads a pre-trained embedding model and converts questions into normalized vector embeddings.
# Normalization helps improve similarity search performance in FAISS using distance metrics.

from sentence_transformers import SentenceTransformer  # Import embedding model

# Load lightweight pre-trained model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate normalized embeddings for all questions
question_embeddings = embed_model.encode(questions, normalize_embeddings=True)

In [ ]:
# This step initializes a FAISS index and stores all question embeddings for fast similarity search.
# It enables efficient retrieval of nearest vectors based on L2 distance.

import faiss  # Library for similarity search
import numpy as np  # Numerical operations

# Get embedding dimension
dimension = question_embeddings.shape[1]

# Create FAISS index using L2 distance
index = faiss.IndexFlatL2(dimension)

# Add embeddings to the index
index.add(np.array(question_embeddings))

# Print total number of stored vectors  
print("FAISS index ready:", index.ntotal)

FAISS index ready: 1326


In [ ]:
# This function retrieves the top-k most similar question-answer pairs using normalized embeddings.
# It improves retrieval accuracy by ensuring consistent similarity comparison in FAISS.

def retrieve(query, k=3):
    # Convert query into normalized embedding
    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True
    )

    # Search for nearest neighbors in FAISS index
    distances, indices = index.search(query_embedding, k)

    # Collect corresponding question-answer pairs
    results = []
    for i in indices[0]:
        results.append((questions[i], answers[i]))

    return results

In [ ]:
# This function builds a structured prompt including context, user answer, and evaluation instructions.
# It guides the LLM to generate consistent evaluation, suggestions, and a score.

def build_prompt(query, retrieved_docs, answer):
    # Combine retrieved Q&A pairs into context
    context = "\n".join([f"Q: {q}\nA: {a}" for q, a in retrieved_docs])

    # Create formatted prompt for evaluation
    prompt = f"""
You are an AI interview evaluator.

Context:
{context}

candidates answer:
{answer}

Question:
{query}

Give output in this format ONLY:
Evaluation:
...
Suggestion:
...
Score: <number>/100
"""
    return prompt

In [ ]:
# This step loads the Mistral LLM and generates evaluation output using retrieved context.
# It integrates retrieval + prompt building + generation into a complete RAG pipeline.

from transformers import AutoTokenizer, AutoModelForCausalLM  # Hugging Face model tools
import torch  # PyTorch for model execution

# Load tokenizer and model
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Use half precision for efficiency
    device_map="auto"  # Automatically use GPU if available
)

def generate_answer(query, answer):
    # Retrieve relevant context
    retrieved = retrieve(query)

    # Build prompt with context and candidate answer
    prompt = build_prompt(query, retrieved, answer)

    # Convert prompt to model input and move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate response from model
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True,
    )

    # Decode output and remove prompt text
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output.replace(prompt, "").strip()

In [ ]:
# This step selects a random question, takes user input, and generates evaluation using the model.
# It simulates an interactive interview practice session.

querys = df["question"].tolist()  # Get list of questions

import random  # For random selection

# Select a random question
query = random.choice(querys)

# Display question
print("Question:", query)

# Take user answer as input
answer = input("\nYour Answer: ")

# Generate and print evaluation
print(generate_answer(query, answer))

Question: What is the role of heuristics in local search algorithms?

Your Answer: A heuristic is a rule-of-thumb or shortcut used to make decisions or solve problems faster, even if the solution is not perfectly optimal.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Evaluation:
The candidate provided a correct definition of heuristics. However, the answer did not specifically address the role of heuristics in local search algorithms.

Suggestion:
The candidate could have provided more detail about how heuristics are used in local search algorithms to guide the search process and improve efficiency.

Score: 50/100
